[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/word_embeddings_sentiment.ipynb)

# Sentiment analysis with word embeddings

We build a sentiment classifier for short movie reviews: each sentence becomes the **average of the GloVe embeddings** of its words, and a **logistic regression** decides if it is positive (1) or negative (0).

The goal is not only accuracy: we want to see **what the embeddings contribute**. The dataset was built for that, with four splits:

| split | content |
|---|---|
| `train` | 400 sentences, 10 positive and 10 negative adjectives |
| `test_seen` | 100 new sentences with the **same** adjectives |
| `test_unseen` | 100 sentences with adjectives that **never appear** in `train` (e.g. *superb*, *dreadful*) |
| `negation` | 20 sentences where a negation flips the meaning (e.g. *not bad at all*) |

All sentences use the same templates and nouns for both classes, so the adjective is the only cue.

## Download GloVe

The ZIP file has ~800 MB, with embeddings of 50, 100, 200 and 300 dimensions. We only extract the 100 dimensions file.

In [ ]:
import os
import urllib.request
import zipfile

glove_dir = "glove"
zip_path = os.path.join(glove_dir, "glove.6B.zip")
glove_name = "glove.6B.100d.txt"
glove_file = os.path.join(glove_dir, glove_name)

os.makedirs(glove_dir, exist_ok=True)

if os.path.exists(glove_file):
    print("GloVe text file already extracted — skipping download.")
else:
    if not os.path.exists(zip_path):
        print("GloVe ZIP not found. Downloading…")
        url = "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"
        urllib.request.urlretrieve(url, zip_path)
        print("Download complete.")

    print("Extracting", glove_name, "…")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extract(glove_name, glove_dir)
    print("Extraction complete.")

## Load the embeddings

Each line of the file has a word followed by the 100 values of its vector (400 000 words).

In [ ]:
import numpy as np

def load_glove(path):
    embeddings = {}
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            embeddings[parts[0]] = np.array(parts[1:], dtype=np.float32)
    return embeddings

glove = load_glove(glove_file)
print("Vocabulary size:", len(glove))

Cosine similarity between some words. Note that *good* and *bad* are **very similar**: antonyms appear in the same contexts (*the movie was good / bad*).

In [ ]:
def cosine(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

for a, b in [("king", "queen"), ("great", "superb"), ("awful", "dreadful"), ("good", "bad")]:
    print(f"sim({a}, {b}) = {cosine(glove[a], glove[b]):.2f}")

## The dataset

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/xiptos/is_notes/refs/heads/main/datasets/movie_sentiment.csv"
df = pd.read_csv(url)

print(df.groupby(["split", "label"]).size().unstack())
df.sample(8, random_state=0)

## From sentences to vectors

We split a sentence into lowercase words, **without punctuation** (`"movie,"` would not be found in GloVe), and average their embeddings:

$$s = \frac{1}{N} \sum_{i=1}^{N} e_{w_i}$$

Words missing from the embeddings are ignored.

In [ ]:
import re

def tokenize(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

def sentence_vector(sentence, emb, dim=100):
    vecs = [emb[w] for w in tokenize(sentence) if w in emb]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

print(tokenize("Honestly, the movie was superb"))
print(sentence_vector("Honestly, the movie was superb", glove).shape)

In [ ]:
def features(split, emb):
    d = df[df.split == split]
    X = np.array([sentence_vector(s, emb) for s in d.sentence])
    return X, d.label.values

X_train, y_train = features("train", glove)
print(X_train.shape, y_train.shape)

## Train and evaluate

The splits are fixed, so there is no `train_test_split` here: every student gets the same result.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

X_seen, y_seen = features("test_seen", glove)
print(f"Accuracy on test_seen: {clf.score(X_seen, y_seen):.2f}")

## Why does it work?

Logistic regression computes $\sigma(\theta^\top s + b)$. Since $s$ is an average,

$$\theta^\top s = \frac{1}{N} \sum_{i=1}^{N} \theta^\top e_{w_i}$$

so every word has a **polarity score** $\theta^\top e_w$, and the sentence score is the average of the scores of its words. The score can be computed for **any** word in GloVe, even if it never appeared in training.

In [ ]:
theta = clf.coef_[0]

for w in ["great", "excellent", "awful", "boring", "movie", "superb", "magnificent", "dreadful", "tedious"]:
    seen = "train" if any(w in tokenize(s) for s in df[df.split == "train"].sentence) else "never seen"
    print(f"{w:>12s}  {theta @ glove[w]:7.2f}   ({seen})")

## Experiment 1: words never seen in training

**Predict first:** what accuracy do you expect on `test_unseen`, whose adjectives never appear in `train`? And if we replace GloVe with **random** vectors?

In [ ]:
X_unseen, y_unseen = features("test_unseen", glove)
print(f"GloVe  — test_seen: {clf.score(X_seen, y_seen):.2f}   test_unseen: {clf.score(X_unseen, y_unseen):.2f}")

Now the same pipeline with a random vector for each word. With only 20 unseen adjectives the result depends on the random draw, so we repeat it with 10 seeds:

In [ ]:
vocab = {w for s in df.sentence for w in tokenize(s)}
acc_seen, acc_unseen = [], []

for seed in range(10):
    rng = np.random.default_rng(seed)
    random_emb = {w: rng.normal(scale=0.4, size=100).astype(np.float32) for w in vocab}

    Xr_train, _ = features("train", random_emb)
    Xr_seen, _ = features("test_seen", random_emb)
    Xr_unseen, _ = features("test_unseen", random_emb)

    clf_random = LogisticRegression(max_iter=1000).fit(Xr_train, y_train)
    acc_seen.append(clf_random.score(Xr_seen, y_seen))
    acc_unseen.append(clf_random.score(Xr_unseen, y_unseen))

print(f"Random — test_seen: {np.mean(acc_seen):.2f}   test_unseen: {np.mean(acc_unseen):.2f} "
      f"(from {min(acc_unseen):.2f} to {max(acc_unseen):.2f})")

**Explain:** why are random vectors as good as GloVe on `test_seen`, but no better than a coin flip on `test_unseen`? What does this tell you about the role of pre-trained embeddings?

## Experiment 2: negation

**Predict first:** how will the classifier handle *The film was not bad at all* or *The movie was far from excellent*?

In [ ]:
neg = df[df.split == "negation"].copy()
X_neg, y_neg = features("negation", glove)
neg["predicted"] = clf.predict(X_neg)
print(f"Accuracy on negation: {clf.score(X_neg, y_neg):.2f}")
neg[["sentence", "label", "predicted"]]

**Explain:** use the polarity scores to explain the errors. Can *any* model that averages word vectors handle *not good* and *good, not bad* correctly?

## Your turn

1. Write a few reviews of your own and classify them with `clf.predict_proba`. Try to fool the classifier.
2. Repeat the experiments with `glove.6B.50d.txt` or `glove.6B.300d.txt`. Does the dimension matter here?
3. In the dataset each template is used for both classes. What could go wrong if, for example, every negative sentence started with *Honestly,*? Check it with the polarity scores.
4. How would you need to change the model to handle negation? (See the `lstm_sentiment` notebook.)

In [ ]:
my_reviews = ["The acting was splendid", "What a waste of time", "The plot was not bad, but the ending was terrible"]
X_mine = np.array([sentence_vector(s, glove) for s in my_reviews])
for s, p in zip(my_reviews, clf.predict_proba(X_mine)[:, 1]):
    print(f"{p:.2f}  {s}")